In [ ]:
!pip install -q sentence-transformers numpy tqdm

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
EMBEDDING_MODEL_NAME = 'sentence-transformers/multi-qa-mpnet-base-cos-v1'

# Dataset to embed. Options:
#   'hotpotqa_distractor'
#   'musique'
#   '2wikimultihopqa'
DATASET = 'hotpotqa_distractor'

# Chunk mode:
#   'sentence'     - one chunk per sentence (default for multi-hop)
#   'document'     - one chunk per document
#   'word_window'  - sliding window of W words, stride = W // 2
#   'token_window' - sliding window of CHUNK_TOKENS words with STRIDE overlap
CHUNK_MODE = 'sentence'

# word_window mode parameters
W = 100  # chunk size in words

# token_window mode parameters
CHUNK_TOKENS = 256  # ch
STRIDE = 128  # st

DRIVE_PATHS = {
    'hotpotqa_distractor': '/content/drive/MyDrive/thesis_data/hotpot_dev_distractor_v1.json',
    'musique': '/content/drive/MyDrive/thesis_data/musique_ans_v1.0_dev.jsonl',
    '2wikimultihopqa': '/content/drive/MyDrive/thesis_data/dev.json',
}
DATASET_PATH = DRIVE_PATHS[DATASET]

BATCH_SIZE = 256

OUTPUT_DIR = f'/content/drive/MyDrive/thesis_cache/{DATASET}/{CHUNK_MODE}'

In [ ]:
# DATASET LOADING

import json
import re


def _split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]


def _word_window_chunks(sentences, w):
    """Sliding window of w whitespace words, stride = w // 2."""
    words = []
    for sent in sentences:
        words.extend(sent.strip().split())
    if not words:
        return []
    stride = w // 2
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + w, len(words))
        chunks.append(' '.join(words[start:end]))
        if end >= len(words):
            break
        start += stride
    return chunks


def _token_window_chunks(sentences, chunk_tokens, stride):
    """
    TODO: replace with real LLM tokenizer for true token counts.
    """
    words = []
    for sent in sentences:
        words.extend(sent.strip().split())
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_tokens, len(words))
        chunks.append(' '.join(words[start:end]))
        if end >= len(words):
            break
        start += stride
    return chunks


def _chunk_doc(sentences, chunk_mode, w, chunk_tokens, stride):
    if chunk_mode == 'sentence':
        return [s.strip() for s in sentences]
    elif chunk_mode == 'document':
        return [' '.join(s.strip() for s in sentences)]
    elif chunk_mode == 'word_window':
        return _word_window_chunks(sentences, w)
    elif chunk_mode == 'token_window':
        return _token_window_chunks(sentences, chunk_tokens, stride)
    else:
        raise ValueError(f'Unknown chunk_mode: {chunk_mode}')


def load_records(dataset, path, chunk_mode, w=100, chunk_tokens=256, stride=128):
    if dataset == 'hotpotqa_distractor':
        return _load_hotpotqa(path, chunk_mode, w, chunk_tokens, stride)
    elif dataset == 'musique':
        return _load_musique(path, chunk_mode, w, chunk_tokens, stride)
    elif dataset == '2wikimultihopqa':
        return _load_2wiki(path, chunk_mode, w, chunk_tokens, stride)
    else:
        raise ValueError(f'Unknown dataset: {dataset}')


def _load_hotpotqa(path, chunk_mode, w, chunk_tokens, stride):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for item in raw:
        chunks = []
        for title, sentences in item['context']:
            chunks.extend(_chunk_doc(sentences, chunk_mode, w, chunk_tokens, stride))
        records.append((item['_id'], item['question'], chunks))
    return records


def _load_musique(path, chunk_mode, w, chunk_tokens, stride):
    records = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            if not item.get('answerable', True):
                continue
            chunks = []
            for para in item['paragraphs']:
                sentences = _split_sentences(para['paragraph_text'])
                chunks.extend(
                    _chunk_doc(sentences, chunk_mode, w, chunk_tokens, stride)
                )
            records.append((item['id'], item['question'], chunks))
    return records


def _load_2wiki(path, chunk_mode, w, chunk_tokens, stride):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for item in raw:
        chunks = []
        for title, sentences in item['context']:
            chunks.extend(_chunk_doc(sentences, chunk_mode, w, chunk_tokens, stride))
        records.append((item['_id'], item['question'], chunks))
    return records


records = load_records(DATASET, DATASET_PATH, CHUNK_MODE, W, CHUNK_TOKENS, STRIDE)
print(f'Loaded {len(records)} records')

In [ ]:
# EMBEDDING

import hashlib
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cuda')
print(f'Model loaded on {model.device}')
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

# cache/<dataset>/<chunk_mode>/<model_hash>/<record_id>.npz
model_hash = hashlib.md5(
    EMBEDDING_MODEL_NAME.encode(), usedforsecurity=False
).hexdigest()[:8]
cache_dir = os.path.join(OUTPUT_DIR, model_hash)

os.makedirs(cache_dir, exist_ok=True)
print(f'Cache dir: {cache_dir}')
print(f'Model hash: {model_hash}')

In [ ]:
# RUN EMBEDDING

skipped = 0
embedded = 0

for record_id, question, chunk_texts in tqdm(records, desc='Embedding'):
    out_path = os.path.join(cache_dir, f'{record_id}.npz')

    # already cached
    if os.path.exists(out_path):
        skipped += 1
        continue

    # encode query + all chunks in one batch
    all_texts = [question] + chunk_texts
    all_embs = model.encode(
        all_texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    query_emb = all_embs[0]
    chunk_embs = all_embs[1:]

    np.savez_compressed(out_path, query=query_emb, chunks=chunk_embs)
    embedded += 1

print(f'Done: {embedded} embedded, {skipped} already cached')
print(f'Cache dir: {cache_dir}')

In [ ]:
# PACKAGE FOR DOWNLOAD

import shutil

zip_name = f'{DATASET}_{CHUNK_MODE}_cache'
zip_path = f'/content/drive/MyDrive/{zip_name}'

shutil.make_archive(zip_path, 'zip', OUTPUT_DIR, '.')
import zipfile

zip_out = f'/content/drive/MyDrive/{zip_name}.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(cache_dir)):
        full = os.path.join(cache_dir, fname)
        arc = os.path.join(DATASET, CHUNK_MODE, model_hash, fname)
        zf.write(full, arc)

size_mb = os.path.getsize(zip_out) / 1024 / 1024
print(f'Zip created: {zip_out} ({size_mb:.1f} MB)')
print()
print('To use locally:')
print(f'  1. Download {zip_name}.zip from Google Drive')
print(f'  2. cd python/')
print(f'  3. unzip {zip_name}.zip -d cache/')
print(f'  4. Run experiments normally — embeddings will be loaded from cache')